# Lichess Puzzle Dataset — Deep Analysis with PySpark

**Project:** Adaptive Chess Puzzle Advisor  
**Author:** Aradevski  
**Date:** 2026-06-25

---

This notebook performs a comprehensive exploratory analysis of the Lichess puzzle database (6M+ puzzles) using PySpark for scalable computation and matplotlib/seaborn for visualisation.

### Sections
1. Environment setup & Spark session
2. Load & schema inspection
3. Missingness & data quality
4. Rating distribution
5. Popularity & engagement analysis
6. Difficulty tier breakdown
7. Theme & tactic category analysis
8. Move length analysis
9. Opening tags analysis
10. Correlation analysis
11. Rating deviation analysis
12. Category co-occurrence
13. Top puzzles by engagement
14. Key findings summary

## 1. Environment Setup & Spark Session

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import itertools

# ── Plotting style ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#2e3348',
    'axes.labelcolor':  '#e8eaf0',
    'xtick.color':      '#9199ad',
    'ytick.color':      '#9199ad',
    'text.color':       '#e8eaf0',
    'grid.color':       '#2e3348',
    'grid.linewidth':   0.5,
    'font.family':      'sans-serif',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})
ACCENT   = '#c9a84c'
GREEN    = '#4caf7d'
RED      = '#e05c5c'
BLUE     = '#5588cc'
PALETTE  = [ACCENT, GREEN, RED, BLUE, '#cc70cc', '#70c8cc', '#e09050']
sns.set_palette(PALETTE)

print('Libraries loaded.')

In [ ]:
spark = (
    SparkSession.builder
    .appName('ChessPuzzleAnalysis')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print(f'Spark {spark.version} ready.')

## 2. Load & Schema Inspection

In [ ]:
CSV_PATH = str(PROJECT_ROOT / 'DataSets' / 'lichess_db_puzzle.csv')

schema = StructType([
    StructField('PuzzleId',        StringType(),  True),
    StructField('FEN',             StringType(),  True),
    StructField('Moves',           StringType(),  True),
    StructField('Rating',          IntegerType(), True),
    StructField('RatingDeviation', IntegerType(), True),
    StructField('Popularity',      IntegerType(), True),
    StructField('NbPlays',         IntegerType(), True),
    StructField('Themes',          StringType(),  True),
    StructField('GameUrl',         StringType(),  True),
    StructField('OpeningTags',     StringType(),  True),
])

df = spark.read.csv(CSV_PATH, header=True, schema=schema)
df.cache()

total = df.count()
print(f'Total puzzles: {total:,}')
df.printSchema()

In [ ]:
df.show(5, truncate=60)

## 3. Missingness & Data Quality

In [ ]:
# Null counts per column
null_counts = df.select([
    F.count(F.when(F.col(c).isNull() | (F.col(c) == ''), c)).alias(c)
    for c in df.columns
]).toPandas().T
null_counts.columns = ['null_count']
null_counts['pct'] = (null_counts['null_count'] / total * 100).round(3)
print(null_counts.sort_values('pct', ascending=False).to_string())

In [ ]:
# Duplicate PuzzleIds
dup_count = total - df.select('PuzzleId').distinct().count()
print(f'Duplicate PuzzleIds: {dup_count:,}')

# Rating sanity
rating_stats = df.select(F.min('Rating'), F.max('Rating'), F.mean('Rating').alias('mean'), F.stddev('Rating').alias('std')).toPandas()
print('\nRating stats:')
print(rating_stats.to_string(index=False))

# Negative popularity
neg_pop = df.filter(F.col('Popularity') < 0).count()
print(f'\nNegative popularity rows: {neg_pop:,}')

# Zero plays
zero_plays = df.filter(F.col('NbPlays') == 0).count()
print(f'Zero-play rows: {zero_plays:,}')

## 4. Rating Distribution

In [ ]:
rating_pd = df.select('Rating').toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(rating_pd['Rating'].dropna(), bins=80, color=ACCENT, edgecolor='none', alpha=0.85)
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Puzzle Rating (Glicko-2)')
axes[0].set_ylabel('Number of Puzzles')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x>=1e6 else f'{x/1e3:.0f}K'))
axes[0].axvline(rating_pd['Rating'].median(), color=GREEN, lw=1.5, linestyle='--', label=f'Median {rating_pd["Rating"].median():.0f}')
axes[0].axvline(rating_pd['Rating'].mean(), color=RED, lw=1.5, linestyle='--', label=f'Mean {rating_pd["Rating"].mean():.0f}')
axes[0].legend()
axes[0].grid(axis='y')

# Box plot by difficulty tier
TIERS = [
    (0,    999,  'Beginner'),
    (1000, 1199, 'Easy'),
    (1200, 1499, 'Intermediate'),
    (1500, 1799, 'Advanced'),
    (1800, 1999, 'Hard'),
    (2000, 2199, 'Expert'),
    (2200, 9999, 'Master'),
]
def assign_tier(r):
    for lo, hi, label in TIERS:
        if lo <= r <= hi:
            return label
    return 'Unknown'

rating_pd['Tier'] = rating_pd['Rating'].apply(assign_tier)
tier_order = [t[2] for t in TIERS]
tier_counts = rating_pd['Tier'].value_counts()[tier_order]
axes[1].barh(tier_order, tier_counts.values, color=PALETTE[:len(TIERS)])
axes[1].set_title('Puzzles per Difficulty Tier')
axes[1].set_xlabel('Number of Puzzles')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x>=1e6 else f'{x/1e3:.0f}K'))
for i, v in enumerate(tier_counts.values):
    axes[1].text(v + 5000, i, f'{v:,}', va='center', fontsize=8, color='#9199ad')
axes[1].grid(axis='x')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Notebooks' / 'fig_rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nRating percentiles:')
print(rating_pd['Rating'].describe(percentiles=[.05,.1,.25,.5,.75,.9,.95]).to_string())

## 5. Popularity & Engagement Analysis

In [ ]:
eng_pd = df.select('Rating', 'Popularity', 'NbPlays', 'RatingDeviation').toPandas()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Popularity histogram
axes[0,0].hist(eng_pd['Popularity'].clip(-100, 100), bins=60, color=BLUE, edgecolor='none', alpha=0.85)
axes[0,0].set_title('Popularity Score Distribution')
axes[0,0].set_xlabel('Popularity (-100 to 100)')
axes[0,0].set_ylabel('Puzzles')
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x>=1e6 else f'{x/1e3:.0f}K'))
axes[0,0].grid(axis='y')

# NbPlays histogram (log scale)
plays_pos = eng_pd['NbPlays'].clip(lower=1)
axes[0,1].hist(np.log10(plays_pos), bins=60, color=GREEN, edgecolor='none', alpha=0.85)
axes[0,1].set_title('Number of Plays (log₁₀ scale)')
axes[0,1].set_xlabel('log₁₀(NbPlays)')
axes[0,1].set_ylabel('Puzzles')
axes[0,1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x>=1e6 else f'{x/1e3:.0f}K'))
axes[0,1].grid(axis='y')

# Rating vs Popularity scatter (sample)
sample = eng_pd.sample(min(30000, len(eng_pd)), random_state=42)
axes[1,0].scatter(sample['Rating'], sample['Popularity'], alpha=0.05, s=1, color=ACCENT)
axes[1,0].set_title('Rating vs Popularity')
axes[1,0].set_xlabel('Rating')
axes[1,0].set_ylabel('Popularity')
axes[1,0].grid()

# Rating vs NbPlays scatter (log)
axes[1,1].scatter(sample['Rating'], np.log10(sample['NbPlays'].clip(lower=1)), alpha=0.05, s=1, color=RED)
axes[1,1].set_title('Rating vs log₁₀(NbPlays)')
axes[1,1].set_xlabel('Rating')
axes[1,1].set_ylabel('log₁₀(NbPlays)')
axes[1,1].grid()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Notebooks' / 'fig_engagement.png', dpi=150, bbox_inches='tight')
plt.show()

print('Popularity stats:')
print(eng_pd['Popularity'].describe().to_string())
print('\nNbPlays stats:')
print(eng_pd['NbPlays'].describe().to_string())

## 6. Difficulty Tier Breakdown (Spark aggregation)

In [ ]:
tier_expr = (
    F.when(F.col('Rating') < 1000, 'Beginner')
     .when(F.col('Rating') < 1200, 'Easy')
     .when(F.col('Rating') < 1500, 'Intermediate')
     .when(F.col('Rating') < 1800, 'Advanced')
     .when(F.col('Rating') < 2000, 'Hard')
     .when(F.col('Rating') < 2200, 'Expert')
     .otherwise('Master')
)

tier_agg = (
    df.withColumn('Tier', tier_expr)
      .groupBy('Tier')
      .agg(
          F.count('*').alias('count'),
          F.mean('Rating').alias('avg_rating'),
          F.mean('Popularity').alias('avg_popularity'),
          F.mean('NbPlays').alias('avg_plays'),
          F.mean('RatingDeviation').alias('avg_rd'),
      )
      .toPandas()
)
tier_agg['order'] = tier_agg['Tier'].map({t[2]: i for i, t in enumerate(TIERS)})
tier_agg = tier_agg.sort_values('order').drop(columns='order').reset_index(drop=True)
tier_agg['pct'] = (tier_agg['count'] / tier_agg['count'].sum() * 100).round(1)
print(tier_agg.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

tier_names = tier_agg['Tier'].tolist()
colors = PALETTE[:len(tier_names)]

axes[0].bar(tier_names, tier_agg['count'], color=colors)
axes[0].set_title('Puzzle Count by Tier')
axes[0].set_ylabel('Puzzles')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x>=1e6 else f'{x/1e3:.0f}K'))
axes[0].tick_params(axis='x', rotation=35)
axes[0].grid(axis='y')

axes[1].bar(tier_names, tier_agg['avg_popularity'], color=colors)
axes[1].set_title('Average Popularity by Tier')
axes[1].set_ylabel('Avg Popularity')
axes[1].tick_params(axis='x', rotation=35)
axes[1].grid(axis='y')

axes[2].bar(tier_names, tier_agg['avg_plays'], color=colors)
axes[2].set_title('Average NbPlays by Tier')
axes[2].set_ylabel('Avg Plays')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K' if x>=1000 else f'{x:.0f}'))
axes[2].tick_params(axis='x', rotation=35)
axes[2].grid(axis='y')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Notebooks' / 'fig_tier_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Theme & Tactic Category Analysis

In [ ]:
# Explode themes into individual rows
theme_df = (
    df.select('PuzzleId', 'Rating', 'Popularity', F.split('Themes', ' ').alias('ThemeList'))
      .withColumn('theme', F.explode('ThemeList'))
      .filter(F.col('theme') != '')
)

# Filter out meta-tags
META_TAGS = {'short','long','veryLong','oneMove','crushing','advantage','equality','master','middlegame','opening','endgame'}
tactic_df = theme_df.filter(~F.col('theme').isin(META_TAGS))

theme_counts = (
    tactic_df.groupBy('theme')
              .agg(
                  F.count('*').alias('count'),
                  F.mean('Rating').alias('avg_rating'),
                  F.mean('Popularity').alias('avg_popularity'),
              )
              .orderBy(F.desc('count'))
              .limit(30)
              .toPandas()
)
print(theme_counts.to_string(index=False))

In [ ]:
top20 = theme_counts.head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Count
axes[0].barh(top20['theme'][::-1], top20['count'][::-1], color=ACCENT)
axes[0].set_title('Top 20 Tactic Themes by Frequency')
axes[0].set_xlabel('Number of Puzzles')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x>=1e6 else f'{x/1e3:.0f}K'))
axes[0].grid(axis='x')

# Avg rating
axes[1].barh(top20['theme'][::-1], top20['avg_rating'][::-1], color=BLUE)
axes[1].set_title('Average Rating per Theme')
axes[1].set_xlabel('Average Puzzle Rating')
axes[1].grid(axis='x')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Notebooks' / 'fig_themes.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Move Length Analysis

In [ ]:
move_df = (
    df.withColumn('move_count', F.size(F.split('Moves', ' ')))
      .withColumn('Tier', tier_expr)
      .select('move_count', 'Rating', 'Tier', 'Popularity')
)

move_agg = (
    move_df.groupBy('move_count')
           .agg(
               F.count('*').alias('count'),
               F.mean('Rating').alias('avg_rating'),
               F.mean('Popularity').alias('avg_popularity')
           )
           .filter(F.col('move_count').between(1, 20))
           .orderBy('move_count')
           .toPandas()
)
print(move_agg.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(move_agg['move_count'], move_agg['count'], color=ACCENT, edgecolor='none')
axes[0].set_title('Puzzle Count by Solution Length')
axes[0].set_xlabel('Number of Moves in Solution')
axes[0].set_ylabel('Puzzles')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x>=1e6 else f'{x/1e3:.0f}K'))
axes[0].grid(axis='y')

axes[1].plot(move_agg['move_count'], move_agg['avg_rating'], color=RED, marker='o', markersize=5)
axes[1].set_title('Avg Rating vs Solution Length')
axes[1].set_xlabel('Number of Moves in Solution')
axes[1].set_ylabel('Average Rating')
axes[1].grid()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Notebooks' / 'fig_move_length.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Opening Tags Analysis

In [ ]:
opening_df = df.filter(F.col('OpeningTags').isNotNull() & (F.col('OpeningTags') != ''))
opening_count = opening_df.count()
print(f'Puzzles with opening tags: {opening_count:,} ({opening_count/total*100:.1f}%)')

top_openings = (
    opening_df
    .withColumn('opening', F.split('OpeningTags', ' ')[0])  # first tag = opening family
    .groupBy('opening')
    .agg(F.count('*').alias('count'), F.mean('Rating').alias('avg_rating'))
    .orderBy(F.desc('count'))
    .limit(20)
    .toPandas()
)
print(top_openings.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top_openings['opening'][::-1], top_openings['count'][::-1], color=GREEN)
ax.set_title('Top 20 Opening Families in Puzzles')
ax.set_xlabel('Puzzle Count')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
ax.grid(axis='x')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Notebooks' / 'fig_openings.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Correlation Analysis

In [ ]:
# Spark correlation matrix
num_cols = ['Rating', 'RatingDeviation', 'Popularity', 'NbPlays']

corr_rows = []
for c1 in num_cols:
    row = []
    for c2 in num_cols:
        r = df.stat.corr(c1, c2)
        row.append(round(r, 4))
    corr_rows.append(row)

corr_df = pd.DataFrame(corr_rows, index=num_cols, columns=num_cols)
print('Pearson correlation matrix:')
print(corr_df.to_string())

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.zeros_like(corr_df.values, dtype=bool)
im = ax.imshow(corr_df.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols))); ax.set_xticklabels(num_cols, rotation=35, ha='right')
ax.set_yticks(range(len(num_cols))); ax.set_yticklabels(num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f'{corr_df.values[i,j]:.2f}', ha='center', va='center', fontsize=11, color='white')
plt.colorbar(im, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Notebooks' / 'fig_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Rating Deviation Analysis

In [ ]:
rd_pd = df.select('RatingDeviation', 'Rating', 'NbPlays').toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(rd_pd['RatingDeviation'].dropna().clip(0, 300), bins=60, color=RED, edgecolor='none', alpha=0.85)
axes[0].set_title('Rating Deviation Distribution')
axes[0].set_xlabel('Rating Deviation (Glicko-2 uncertainty)')
axes[0].set_ylabel('Puzzles')
axes[0].axvline(150, color=ACCENT, lw=1.5, linestyle='--', label='Quality cutoff (RD=150)')
axes[0].legend()
axes[0].grid(axis='y')

sample = rd_pd.sample(min(20000, len(rd_pd)), random_state=42)
axes[1].scatter(np.log10(sample['NbPlays'].clip(lower=1)), sample['RatingDeviation'].clip(0, 300),
                alpha=0.08, s=1, color=BLUE)
axes[1].set_title('NbPlays vs Rating Deviation')
axes[1].set_xlabel('log₁₀(NbPlays)')
axes[1].set_ylabel('Rating Deviation')
axes[1].grid()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Notebooks' / 'fig_rating_deviation.png', dpi=150, bbox_inches='tight')
plt.show()

high_rd = (rd_pd['RatingDeviation'] > 150).sum()
print(f'Puzzles with RD > 150 (poor calibration): {high_rd:,} ({high_rd/len(rd_pd)*100:.1f}%)')

## 12. Category Co-occurrence

In [ ]:
# Pull themes as Python list for co-occurrence (sampled)
themes_sample = (
    df.select(F.split('Themes', ' ').alias('themes'))
      .sample(fraction=0.05, seed=42)
      .toPandas()
)

TACTIC_TAGS = {
    'fork','pin','skewer','discoveredAttack','hangingPiece','sacrifice',
    'deflection','attraction','interference','clearance','quietMove','zugzwang',
    'xRayAttack','kingsafety','mateIn1','mateIn2','mateIn3','mateIn4','mateIn5',
    'backRankMate','promotion','enPassant'
}

cooccur = Counter()
tag_count = Counter()
for tags in themes_sample['themes']:
    tactic_tags = [t for t in tags if t in TACTIC_TAGS]
    for t in tactic_tags:
        tag_count[t] += 1
    for pair in itertools.combinations(sorted(set(tactic_tags)), 2):
        cooccur[pair] += 1

top_tags = [t for t, _ in tag_count.most_common(12)]
matrix = pd.DataFrame(0, index=top_tags, columns=top_tags)
for (a, b), count in cooccur.items():
    if a in top_tags and b in top_tags:
        matrix.loc[a, b] = count
        matrix.loc[b, a] = count

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(matrix.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(top_tags))); ax.set_xticklabels(top_tags, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(top_tags))); ax.set_yticklabels(top_tags, fontsize=9)
plt.colorbar(im, ax=ax, label='Co-occurrence count (5% sample)')
ax.set_title('Tactic Theme Co-occurrence Heatmap')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Notebooks' / 'fig_cooccurrence.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 15 theme pairs:')
for (a, b), c in cooccur.most_common(15):
    print(f'  {a} + {b}: {c:,}')

## 13. Top Puzzles by Engagement

In [ ]:
# Engagement score = NbPlays * (Popularity / 100)
top_puzzles = (
    df.withColumn('engagement', F.col('NbPlays') * (F.col('Popularity') / 100))
      .filter(F.col('Popularity') > 0)
      .orderBy(F.desc('engagement'))
      .select('PuzzleId', 'Rating', 'Popularity', 'NbPlays', 'engagement', 'Themes', 'GameUrl')
      .limit(20)
      .toPandas()
)
top_puzzles['engagement'] = top_puzzles['engagement'].round(0).astype(int)
print('Top 20 most engaged puzzles:')
print(top_puzzles[['PuzzleId','Rating','Popularity','NbPlays','engagement','Themes']].to_string(index=False))

In [ ]:
# Rating distribution of top puzzles
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(top_puzzles['PuzzleId'], top_puzzles['NbPlays'], color=ACCENT)
ax.set_title('Top 20 Puzzles by NbPlays')
ax.set_xlabel('Puzzle ID')
ax.set_ylabel('Number of Plays')
ax.tick_params(axis='x', rotation=60)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x>=1e6 else f'{x/1e3:.0f}K'))
ax.grid(axis='y')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Notebooks' / 'fig_top_puzzles.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Key Findings Summary

In [ ]:
rating_stats_pd = df.select(
    F.count('Rating').alias('n'),
    F.mean('Rating').alias('mean'),
    F.expr('percentile_approx(Rating, 0.5)').alias('median'),
    F.stddev('Rating').alias('std'),
    F.min('Rating').alias('min'),
    F.max('Rating').alias('max'),
).toPandas()

print('=' * 60)
print('DATASET KEY FINDINGS')
print('=' * 60)
print(f'Total puzzles          : {total:>12,}')
print(f'Rating range           : {int(rating_stats_pd["min"].iloc[0]):>6} – {int(rating_stats_pd["max"].iloc[0])}')
print(f'Mean rating            : {rating_stats_pd["mean"].iloc[0]:>12.1f}')
print(f'Median rating          : {rating_stats_pd["median"].iloc[0]:>12.0f}')
print(f'Rating std dev         : {rating_stats_pd["std"].iloc[0]:>12.1f}')
print()
print('Largest tactic categories (by theme tag):')
for _, row in theme_counts.head(8).iterrows():
    print(f'  {row["theme"]:<25} {int(row["count"]):>9,} puzzles  avg rating {row["avg_rating"]:.0f}')
print()
print('Difficulty distribution:')
for _, row in tier_agg.iterrows():
    print(f'  {row["Tier"]:<15} {int(row["count"]):>9,} puzzles  ({row["pct"]}%)')
print('=' * 60)

In [ ]:
spark.stop()
print('Spark session closed.')